In [406]:
%pip install psycopg2

Note: you may need to restart the kernel to use updated packages.


In [55]:
import importlib.util
import sys
from pathlib import Path

# Define the path to the file
file_path = Path("../../../../llama-index-integrations/graph_stores/llama-index-graph-stores-agensgraph/llama_index/graph_stores/agensgraph/agensgraph_property_graph.py").resolve()

# Load the module dynamically
spec = importlib.util.spec_from_file_location("agensgraph_property_graph", str(file_path))
module = importlib.util.module_from_spec(spec)
sys.modules["agensgraph_property_graph"] = module
spec.loader.exec_module(module)

# Now, import the class
AgensGraph = module.AgensPropertyGraphStore

In [56]:
conf = {
    "database": "test",
    "user": "taha-linux",
    "password": "",
    "host": "localhost",
    "port": 5432,
}

# If you want to create a new graph, add the argument `create=True` to the constructor
graph_store = AgensGraph(graph_name="test183", conf=conf, create=True)


            SELECT ARRAY(
                    SELECT labname 
                    FROM ag_label 
                    WHERE labkind = 'e' 
                    AND graphid = 17311
                    AND labname NOT IN ('ag_edge')
                ) as labels;
            

            SELECT labels FROM label_catalog
            WHERE graph_id = 17311
            

    MATCH (start_node)-[r]->(end_node)
    WITH DISTINCT start_node.labels AS start_labels, type(r) AS relationship_type, end_node.labels AS end_labels
    UNWIND start_labels AS start_label
    UNWIND end_labels AS end_label
    WITH DISTINCT start_label, relationship_type, end_label
    WHERE start_label != '__Entity__' AND end_label != '__Entity__'
    RETURN {start: start_label, type: relationship_type, end: end_label} AS output

CREATE CONSTRAINT ON "__Node__"
                        ASSERT id IS UNIQUE;


In [57]:
from llama_index.core.graph_stores.types import EntityNode, ChunkNode, Relation

### GET

In [58]:
entity = EntityNode(label="PERSON", name="Alice")
chunk = ChunkNode(text="Alice is a software engineer.")
graph_store.upsert_nodes([entity, chunk])

graph_store.get(ids=[entity.id])
graph_store.get(properties={"name": "Alice"})


                    UNWIND [{'label': 'text_chunk', 'properties': {}, 'text': 'Alice is a software engineer.', 'id': '8369762666020800263'}] AS row
                    MERGE (c:"__Node__" {id: row.id})
                    SET c.text = row.text, c.labels = append_label(c.labels, 'Chunk')
                    WITH c, row
                    SET c += row.properties, c.embedding = row.embedding
                    RETURN count(*)
                    

                    UNWIND [{'label': 'PERSON', 'properties': {}, 'name': 'Alice', 'id': 'Alice'}] AS row
                    MERGE (e:"__Node__" {id: row.id})
                    SET e += CASE WHEN row.properties IS NOT NULL THEN row.properties ELSE properties(e) END
                    SET e.name = CASE WHEN row.name IS NOT NULL THEN row.name ELSE e.name END,
                        e.labels = append_label(e.labels, '__Entity__')
                    WITH e, row
                    SET e.labels = append_label(e.labels, row.label);

         

[EntityNode(label='PERSON', embedding=None, properties={'name': 'Alice'}, name='Alice')]

### get_triplets

In [59]:
person = EntityNode(label="PERSON", name="Alice")
city = EntityNode(label="CITY", name="Paris")
graph_store.upsert_nodes([person, city])

visited_relation = Relation(
    source_id=person.id,
    target_id=city.id,
    label="VISITED",
    properties={"year": 2023},
)
graph_store.upsert_relations([visited_relation])

graph_store.get_triplets(entity_names=["Alice"])


                    UNWIND [{'label': 'PERSON', 'properties': {}, 'name': 'Alice', 'id': 'Alice'}, {'label': 'CITY', 'properties': {}, 'name': 'Paris', 'id': 'Paris'}] AS row
                    MERGE (e:"__Node__" {id: row.id})
                    SET e += CASE WHEN row.properties IS NOT NULL THEN row.properties ELSE properties(e) END
                    SET e.name = CASE WHEN row.name IS NOT NULL THEN row.name ELSE e.name END,
                        e.labels = append_label(e.labels, '__Entity__')
                    WITH e, row
                    SET e.labels = append_label(e.labels, row.label);

                    UNWIND [{'label': 'PERSON', 'properties': {}, 'name': 'Alice', 'id': 'Alice'}, {'label': 'CITY', 'properties': {}, 'name': 'Paris', 'id': 'Paris'}] AS row
                    MATCH (e:"__Node__" {id: row.id})
                    WHERE row.embedding IS NOT NULL
                    SET e.embedding = row.embedding
                    WITH e, row
                    WHERE 

[[EntityNode(label='PERSON', embedding=None, properties={'id': 'Alice'}, name='Alice'),
  Relation(label='VISITED', source_id='Alice', target_id='Paris', properties={'year': 2023}),
  EntityNode(label='CITY', embedding=None, properties={'id': 'Paris'}, name='Paris')]]

### get_rel_map()

In [60]:
e1 = EntityNode(label="PERSON", name="Alice")
e2 = EntityNode(label="PERSON", name="Bob")
e3 = EntityNode(label="CITY", name="Paris")
e4 = EntityNode(label="CITY", name="London")
graph_store.upsert_nodes([e1, e2, e3, e4])

r1 = Relation(label="KNOWS", source_id=e1.id, target_id=e2.id)
r2 = Relation(label="VISITED", source_id=e1.id, target_id=e3.id)
r3 = Relation(label="VISITED", source_id=e2.id, target_id=e4.id)
graph_store.upsert_relations([r1, r2, r3])

graph_store.get_rel_map([e1, e2], depth=2)


                    UNWIND [{'label': 'PERSON', 'properties': {}, 'name': 'Alice', 'id': 'Alice'}, {'label': 'PERSON', 'properties': {}, 'name': 'Bob', 'id': 'Bob'}, {'label': 'CITY', 'properties': {}, 'name': 'Paris', 'id': 'Paris'}, {'label': 'CITY', 'properties': {}, 'name': 'London', 'id': 'London'}] AS row
                    MERGE (e:"__Node__" {id: row.id})
                    SET e += CASE WHEN row.properties IS NOT NULL THEN row.properties ELSE properties(e) END
                    SET e.name = CASE WHEN row.name IS NOT NULL THEN row.name ELSE e.name END,
                        e.labels = append_label(e.labels, '__Entity__')
                    WITH e, row
                    SET e.labels = append_label(e.labels, row.label);

                    UNWIND [{'label': 'PERSON', 'properties': {}, 'name': 'Alice', 'id': 'Alice'}, {'label': 'PERSON', 'properties': {}, 'name': 'Bob', 'id': 'Bob'}, {'label': 'CITY', 'properties': {}, 'name': 'Paris', 'id': 'Paris'}, {'label': 'CITY', 

[[EntityNode(label='PERSON', embedding=None, properties={'name': 'Alice'}, name='Alice'),
  Relation(label='VISITED', source_id='Alice', target_id='Paris', properties={'year': 2023}),
  EntityNode(label='CITY', embedding=None, properties={'name': 'Paris'}, name='Paris')],
 [EntityNode(label='PERSON', embedding=None, properties={'name': 'Bob'}, name='Bob'),
  Relation(label='VISITED', source_id='Bob', target_id='London', properties={}),
  EntityNode(label='CITY', embedding=None, properties={'name': 'London'}, name='London')],
 [EntityNode(label='PERSON', embedding=None, properties={'name': 'Alice'}, name='Alice'),
  Relation(label='KNOWS', source_id='Alice', target_id='Bob', properties={}),
  EntityNode(label='PERSON', embedding=None, properties={'name': 'Bob'}, name='Bob')],
 [EntityNode(label='PERSON', embedding=None, properties={'name': 'Alice'}, name='Alice'),
  Relation(label='VISITED', source_id='Alice', target_id='Paris', properties={'year': 2023}),
  EntityNode(label='CITY', emb

In [61]:
from llama_index.core.vector_stores.types import VectorStoreQuery

entity1 = EntityNode(
    label="PERSON", name="Alice", properties={"embedding": [0.1, 0.2, 0.3]}
)
entity2 = EntityNode(
    label="PERSON", name="Bob", properties={"embedding": [0.9, 0.8, 0.7]}
)
graph_store.upsert_nodes([entity1, entity2])

# # Query embedding somewhat closer to [0.1, 0.2, 0.3] than [0.9, 0.8, 0.7]
query = VectorStoreQuery(query_embedding=[0.1, 0.2, 0.31], similarity_top_k=2)
graph_store.vector_query(query)


                    UNWIND [{'label': 'PERSON', 'properties': {'embedding': [0.1, 0.2, 0.3]}, 'name': 'Alice', 'id': 'Alice'}, {'label': 'PERSON', 'properties': {'embedding': [0.9, 0.8, 0.7]}, 'name': 'Bob', 'id': 'Bob'}] AS row
                    MERGE (e:"__Node__" {id: row.id})
                    SET e += CASE WHEN row.properties IS NOT NULL THEN row.properties ELSE properties(e) END
                    SET e.name = CASE WHEN row.name IS NOT NULL THEN row.name ELSE e.name END,
                        e.labels = append_label(e.labels, '__Entity__')
                    WITH e, row
                    SET e.labels = append_label(e.labels, row.label);

                    UNWIND [{'label': 'PERSON', 'properties': {'embedding': [0.1, 0.2, 0.3]}, 'name': 'Alice', 'id': 'Alice'}, {'label': 'PERSON', 'properties': {'embedding': [0.9, 0.8, 0.7]}, 'name': 'Bob', 'id': 'Bob'}] AS row
                    MATCH (e:"__Node__" {id: row.id})
                    WHERE row.embedding IS NOT NULL
  

([EntityNode(label='PERSON', embedding=None, properties={}, name='Alice'),
  EntityNode(label='PERSON', embedding=None, properties={}, name='Bob')],
 [0.9998778131472615, 0.8771844461223388])